# 03 — QLoRA Fine-Tuning (Local GPU)

**Purpose:** Run QLoRA training locally on an Ada / Ampere GPU (≥ 20 GB VRAM).

All training logic lives in `src/training/train.py`.  
This notebook orchestrates the two runs and generates report figures.

**Requirements:**
- NVIDIA Ada or Ampere GPU with ≥ 16 GB VRAM (tested on 20 GB Ada)
- `flash-attn` installed (STEP 2) — provides ~3–4× speedup over eager attention
- `HF_TOKEN` and `WANDB_API_KEY` in a `.env` file at the repo root (STEP 3)

## STEP 1 — Environment setup and path resolution

In [ ]:
import os
import sys
from pathlib import Path

# Resolve repo root whether notebook is opened from repo root or notebooks/
REPO_ROOT = Path(os.getcwd())
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

PROCESSED_DIR  = REPO_ROOT / 'data' / 'processed'
CHECKPOINT_DIR = REPO_ROOT / 'checkpoints'
FIGURES_DIR    = REPO_ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'REPO_ROOT : {REPO_ROOT}')
print(f'PROCESSED : {PROCESSED_DIR}')
print(f'FIGURES   : {FIGURES_DIR}')

## STEP 2 — Install flash-attn (skip if already installed)

`flash-attn` compiles a CUDA extension on first install (~2–5 min).  
Run this cell once; subsequent runs skip it automatically.

In [ ]:
try:
    import flash_attn
    print(f'flash-attn {flash_attn.__version__} already installed — skipping')
except ImportError:
    print('Installing flash-attn (compiles CUDA extension, may take a few minutes)...')
    !pip install flash-attn --no-build-isolation -q
    print('Done.')

## STEP 3 — Load secrets from `.env`

Create `<repo_root>/.env` with:
```
HF_TOKEN=hf_...
WANDB_API_KEY=...
```
The file is git-ignored. Alternatively, set the env vars in your shell before launching Jupyter.

In [ ]:
env_file = REPO_ROOT / '.env'
if env_file.exists():
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())
    print(f'Loaded secrets from {env_file}')
else:
    print(f'No .env found at {env_file} — using existing environment variables')

import huggingface_hub
hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    huggingface_hub.login(token=hf_token, add_to_git_credential=False)
    print('HuggingFace: logged in')
else:
    print('WARNING: HF_TOKEN not set — needed for google/medgemma-4b-it (gated)')

import wandb
wandb_key = os.environ.get('WANDB_API_KEY')
if wandb_key:
    wandb.login(key=wandb_key)
    print('W&B: logged in')
else:
    print('WARNING: WANDB_API_KEY not set — training will log to W&B anonymously or offline')

## STEP 4 — Verify data and GPU

In [ ]:
import logging
import torch
import yaml
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

with open(REPO_ROOT / 'params.yaml') as f:
    params = yaml.safe_load(f)

# GPU info
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        cc = f'{props.major}.{props.minor}'
        fa2 = 'flash-attn2 OK' if props.major >= 8 else 'no flash-attn2 (need cc>=8.0)'
        print(f'GPU {i}: {props.name}  {props.total_memory/1e9:.1f} GB  cc={cc}  [{fa2}]')
else:
    print('WARNING: No CUDA GPU found')

# Parquet checks
for split in ('train', 'val', 'test'):
    p = PROCESSED_DIR / f'{split}.parquet'
    if p.exists():
        df = pd.read_parquet(p)
        print(f'{split}.parquet: {len(df)} studies — OK')
    else:
        print(f'MISSING: {p} — run src.data.load / labels / split first')

# Image spot-check (default local path)
images_dir = REPO_ROOT / params['data']['images_dir'] / 'images_normalized'
train_df = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
has_frontal = train_df['frontal'].apply(lambda x: hasattr(x, '__len__') and len(x) > 0)
print(f'Studies with frontal image: {has_frontal.sum()}/{len(train_df)}')
if has_frontal.sum() > 0:
    sample = list(train_df[has_frontal].iloc[0]['frontal'])[0]
    img_path = images_dir / sample
    print(f'Image spot-check: {img_path}  (exists={img_path.exists()})')
    if not img_path.exists():
        print('WARNING: images not found — check params.yaml data.images_dir')

## STEP 5 — Run 1: Uniform sampling (no shift correction)

Flash Attention 2 + VRAM-aware batch scaling are applied automatically by `train.py`.  
Expected time on 20 GB Ada: **~3–4 h** for 3 epochs.

In [ ]:
import json

UNIFORM_RESULTS = CHECKPOINT_DIR / 'qlora_uniform' / 'training_results.json'

if UNIFORM_RESULTS.exists():
    with open(UNIFORM_RESULTS) as f:
        r = json.load(f)
    print(f'Uniform run already complete — best_epoch={r["best_epoch"]}  '
          f'micro_F1={r["best_val_f1_chexbert_micro"]:.4f}')
    RUN_UNIFORM = False
else:
    RUN_UNIFORM = True
    print('Will run uniform training...')

In [ ]:
if RUN_UNIFORM:
    !python -m src.training.train --sampler uniform --run_name qlora_uniform

## STEP 6 — Run 2: Importance-weighted sampling (shift correction)

In [ ]:
WEIGHTED_RESULTS = CHECKPOINT_DIR / 'qlora_weighted' / 'training_results.json'

if WEIGHTED_RESULTS.exists():
    with open(WEIGHTED_RESULTS) as f:
        r = json.load(f)
    print(f'Weighted run already complete — best_epoch={r["best_epoch"]}  '
          f'micro_F1={r["best_val_f1_chexbert_micro"]:.4f}')
    RUN_WEIGHTED = False
else:
    RUN_WEIGHTED = True
    print('Will run weighted training...')

In [ ]:
if RUN_WEIGHTED:
    !python -m src.training.train --sampler weighted --run_name qlora_weighted

## STEP 7 — Load training results

In [ ]:
import json
import numpy as np
import pandas as pd

results = {}
for variant in ('uniform', 'weighted'):
    rpath = CHECKPOINT_DIR / f'qlora_{variant}' / 'training_results.json'
    if rpath.exists():
        with open(rpath) as f:
            results[variant] = json.load(f)
        print(f'[{variant}] best_epoch={results[variant]["best_epoch"]}  '
              f'best_micro_F1={results[variant]["best_val_f1_chexbert_micro"]:.4f}')
    else:
        print(f'[{variant}] results not found — run STEP 5/6 first')

baseline_path = REPO_ROOT / 'reports' / 'baseline_results.json'
baseline_results = json.loads(baseline_path.read_text()) if baseline_path.exists() else None
if baseline_results:
    m = baseline_results['metrics']
    print(f'[zero-shot] micro F1={m["f1_chexbert_micro_present"]:.4f}  '
          f'macro F1={m["f1_chexbert_macro_present"]:.4f}')
else:
    print('[zero-shot] baseline_results.json not found — run notebook 02 first')

## STEP 8 — Figure: Training loss comparison

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 120

fig, ax = plt.subplots(figsize=(11, 4))

colors = {'uniform': '#1f77b4', 'weighted': '#d62728'}
for variant, color in colors.items():
    if variant not in results:
        continue
    log = results[variant].get('log_history', [])
    steps  = [e['step']  for e in log if 'loss' in e]
    losses = [e['loss']  for e in log if 'loss' in e]
    if steps:
        ax.plot(steps, losses, color=color, linewidth=1.5, label=f'sampler={variant}', alpha=0.9)

ax.set_xlabel('Training step')
ax.set_ylabel('Loss (causal LM)')
ax.set_title('QLoRA Training Loss — Uniform vs Weighted Sampler')
ax.legend(fontsize=11)
plt.tight_layout()

fig_path = FIGURES_DIR / 'train_loss_comparison.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved {fig_path}')
plt.show()

## STEP 9 — Figure: Val F1-CheXbert per epoch

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

metric_names = ['val_f1_chexbert_micro', 'val_f1_chexbert_macro']
titles       = ['Micro F1-CheXbert', 'Macro F1-CheXbert']
zs_values    = [None, None]
if baseline_results:
    m = baseline_results['metrics']
    zs_values = [m['f1_chexbert_micro_present'], m['f1_chexbert_macro_present']]

colors_v = {'uniform': '#1f77b4', 'weighted': '#d62728'}
markers  = {'uniform': 'o', 'weighted': 's'}

for ax, metric, title, zs_val in zip(axes, metric_names, titles, zs_values):
    for variant in ('uniform', 'weighted'):
        if variant not in results:
            continue
        hist = results[variant].get('history', [])
        epochs = [r['epoch'] for r in hist]
        vals   = [r[metric]  for r in hist]
        ax.plot(epochs, vals, marker=markers[variant], color=colors_v[variant],
                linewidth=2, label=f'{variant}', markersize=8)

    if zs_val is not None:
        ax.axhline(zs_val, color='gray', linestyle='--', linewidth=1.2,
                   label=f'zero-shot = {zs_val:.3f}')

    ax.set_xlabel('Epoch')
    ax.set_ylabel('F1')
    ax.set_title(title)
    ax.set_ylim(0, 1.0)
    ax.legend(fontsize=10)

fig.suptitle('Val F1-CheXbert per Epoch — QLoRA Fine-Tuning', fontsize=13)
plt.tight_layout()

fig_path = FIGURES_DIR / 'train_val_f1_comparison.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved {fig_path}')
plt.show()

## STEP 10 — Figure: Sampler weight distribution

In [ ]:
from src.data.labels import CHEXBERT_LABELS
from src.training.sampler import build_sample_weights

train_df = pd.read_parquet(PROCESSED_DIR / 'train.parquet')
train_df = train_df[train_df['findings'].notna() & (train_df['findings'].str.strip() != '')]
label_matrix = train_df[CHEXBERT_LABELS].values.astype(float)

p_target    = params['sampler'].get('p_target', {})
weight_clip = float(params['sampler'].get('weight_clip', 10.0))

weights_uniform  = build_sample_weights(label_matrix, p_target=None, weight_clip=weight_clip)
weights_weighted = build_sample_weights(label_matrix, p_target=p_target, weight_clip=weight_clip)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, weights, title, color in [
    (axes[0], weights_uniform,  'Uniform sampler weights', '#1f77b4'),
    (axes[1], weights_weighted, 'Importance weights (weighted sampler)', '#d62728'),
]:
    ax.hist(weights, bins=60, color=color, alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.axvline(weights.mean(), color='black', linestyle='--', linewidth=1.5,
               label=f'mean={weights.mean():.2f}')
    ax.axvline(np.median(weights), color='red', linestyle=':', linewidth=1.5,
               label=f'median={np.median(weights):.2f}')
    ax.set_xlabel('Sample weight')
    ax.set_ylabel('Training studies')
    ax.set_title(title)
    ax.legend(fontsize=10)

ess = weights_weighted.sum()**2 / (weights_weighted**2).sum()
n   = len(weights_weighted)
axes[1].set_title(f'Importance weights — ESS={ess:.0f}/{n} ({100*ess/n:.1f}%)')

plt.tight_layout()
fig_path = FIGURES_DIR / 'train_sampler_weights_comparison.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved {fig_path}')
plt.show()

## STEP 11 — Figure: Per-label F1 comparison

In [ ]:
def _best_per_label(results_dict):
    hist = results_dict.get('history', [])
    if not hist:
        return {}
    best = max(hist, key=lambda r: r['val_f1_chexbert_micro'])
    return best.get('per_label_f1', {})

series = {}
if baseline_results:
    series['zero-shot'] = pd.Series(baseline_results['metrics']['per_label_f1_present'])
for variant in ('uniform', 'weighted'):
    if variant in results:
        pl = _best_per_label(results[variant])
        if pl:
            series[variant] = pd.Series(pl)

if not series:
    print('No results to plot — run training first.')
else:
    df_plot = pd.DataFrame(series).reindex(CHEXBERT_LABELS).fillna(0)
    x = np.arange(len(CHEXBERT_LABELS))
    n_cols = len(df_plot.columns)
    w = 0.8 / n_cols
    col_colors = {'zero-shot': '#aec7e8', 'uniform': '#1f77b4', 'weighted': '#d62728'}

    fig, ax = plt.subplots(figsize=(14, 6))
    for i, col in enumerate(df_plot.columns):
        offset = (i - n_cols / 2 + 0.5) * w
        ax.bar(x + offset, df_plot[col].values, w,
               label=col, color=col_colors.get(col, f'C{i}'), alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(CHEXBERT_LABELS, rotation=40, ha='right', fontsize=9)
    ax.set_ylabel('F1-CheXbert')
    ax.set_ylim(0, 1.0)
    ax.set_title('Per-Label F1-CheXbert: Zero-Shot vs Fine-Tuned (Uniform vs Weighted)')
    ax.legend(fontsize=11)
    plt.tight_layout()

    fig_path = FIGURES_DIR / 'train_per_label_f1_comparison.png'
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f'Saved {fig_path}')
    plt.show()

## STEP 12 — Metric summary table

In [ ]:
rows = []

if baseline_results:
    m = baseline_results['metrics']
    rows.append({
        'Condition'         : 'Zero-shot (no fine-tune)',
        'Sampler'           : '—',
        'Best epoch'        : '—',
        'Micro F1 (present)': round(m['f1_chexbert_micro_present'], 4),
        'Macro F1 (present)': round(m['f1_chexbert_macro_present'], 4),
        'BERTScore F1'      : round(m.get('bertscore_f1', float('nan')), 4),
    })

for variant in ('uniform', 'weighted'):
    if variant not in results:
        continue
    r = results[variant]
    best_hist = max(r.get('history', [{}]), key=lambda x: x.get('val_f1_chexbert_micro', -1))
    rows.append({
        'Condition'         : 'QLoRA fine-tune',
        'Sampler'           : variant,
        'Best epoch'        : r.get('best_epoch', '—'),
        'Micro F1 (present)': round(best_hist.get('val_f1_chexbert_micro', float('nan')), 4),
        'Macro F1 (present)': round(best_hist.get('val_f1_chexbert_macro', float('nan')), 4),
        'BERTScore F1'      : float('nan'),
    })

if rows:
    summary_df = pd.DataFrame(rows).set_index(['Condition', 'Sampler'])
    display(summary_df)

    fig, ax = plt.subplots(figsize=(12, 1.5 + 0.6 * len(rows)))
    ax.axis('off')
    summary_df_reset = summary_df.reset_index()
    tbl = ax.table(
        cellText=summary_df_reset.values,
        colLabels=summary_df_reset.columns.tolist(),
        cellLoc='center', loc='center',
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1.0, 1.8)
    for j in range(len(summary_df_reset.columns)):
        tbl[(0, j)].set_facecolor('#2c5f8a')
        tbl[(0, j)].set_text_props(color='white', fontweight='bold')
    ax.set_title('QLoRA Training — Metric Summary (val set)', fontsize=12, pad=12)
    plt.tight_layout()

    fig_path = FIGURES_DIR / 'train_metric_summary.png'
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f'Saved {fig_path}')
    plt.show()
else:
    print('No results to display — run training first.')

## Done

Best checkpoints are in:
```
checkpoints/qlora_uniform/best_model/
checkpoints/qlora_weighted/best_model/
```

Figures saved to `reports/figures/`:

| Figure | Content |
|--------|---------|
| `train_loss_comparison.png` | Training loss curves |
| `train_val_f1_comparison.png` | Val F1-CheXbert per epoch |
| `train_sampler_weights_comparison.png` | Uniform vs importance weight distribution |
| `train_per_label_f1_comparison.png` | Per-label F1: zero-shot vs both fine-tuned variants |
| `train_metric_summary.png` | Summary table |

**Next step:** `notebooks/04_eval_and_figures.ipynb` — full metric stack on test set.